# Lab 7 — Support Vector Machines (SVM)
## Haberman Dataset

**Donghyeon & Dae Young**

**Dataset Source:** https://archive.ics.uci.edu/dataset/43/haberman+s+survival

In [ ]:
!pip -q install ucimlrepo

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from ucimlrepo import fetch_ucirepo

In [ ]:
import pandas as pd

df = pd.read_csv(
    'haberman.data',
    header=None,
    names=['age', 'operation_year', 'positive_nodes', 'survival_status']
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (306, 4)
   age  operation_year  positive_nodes  survival_status
0   30              64               1                1
1   30              62               3                1
2   30              65               0                1
3   31              59               2                1
4   31              65               4                1


In [ ]:
# No text columns need to be removed for this dataset.
# Unlike Advertising, all three features are already numeric.
print('Missing values:', df.isna().sum().sum())

Missing values: 0


In [ ]:
# No LabelEncoder is needed; features and target are numeric already.
print('Target distribution:')
print(df['survival_status'].value_counts().sort_index())

Target distribution:
survival_status
1    225
2     81
Name: count, dtype: int64


In [ ]:
df.head()

,age,operation_year,positive_nodes,survival_status
0,30,64,1,1
1,30,62,3,1
2,30,65,0,1
3,31,59,2,1
4,31,65,4,1


In [ ]:
# Split between features as X and target as y
X = df.drop('survival_status', axis=1)
y = df['survival_status']
print('Features:', X.columns.tolist())

Features: ['age', 'operation_year', 'positive_nodes']


In [ ]:
# Set the data into training and test sets
# Keeping the same test_size and random_state as the professor's example
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Apply StandardScaler to the input features
# Fit only on training data so that test data does not leak into training
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Set the model
model = SVC()

# Use One-vs-Rest (OvA) strategy
# model = OneVsRestClassifier(SVC())

# Use One-vs-One (OvO) strategy
# model = OneVsOneClassifier(SVC())

In [ ]:
# Model fit for the training data
model.fit(X_train, y_train)

SVC()

In [ ]:
# Make predictions on the test data
model_predict = model.predict(X_test)

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, model_predict)
print('\nAccuracy:', accuracy)


Accuracy: 0.6612903225806451


In [ ]:
# Run precision, recall, and F1-score
report = classification_report(y_test, model_predict, zero_division=0)
print('\nClassification Report:\n', report)

# Macro-average gives equal weight to the two classes
precision = precision_score(y_test, model_predict, average='macro', zero_division=0)
recall = recall_score(y_test, model_predict, average='macro', zero_division=0)
f1 = f1_score(y_test, model_predict, average='macro', zero_division=0)

print('\nMacro Precision:', precision)
print('Macro Recall:', recall)
print('Macro F1-Score:', f1)


Classification Report:
               precision    recall  f1-score   support

           1       0.73      0.87      0.79        46
           2       0.14      0.06      0.09        16

    accuracy                           0.66        62
   macro avg       0.44      0.47      0.44        62
weighted avg       0.58      0.66      0.61        62


Macro Precision: 0.43506493506493504
Macro Recall: 0.46603260869565216
Macro F1-Score: 0.43951786482996125


In [ ]:
def get_new_data_point():
    """Ask for the three numeric features and allow retrying invalid input."""
    while True:
        try:
            age = float(input('Enter age at surgery (years): '))
            operation_year = float(input('Enter operation year minus 1900 (e.g., 64 for 1964): '))
            positive_nodes = float(input('Enter number of positive axillary nodes: '))
            if not (0 <= age <= 120 and 0 <= operation_year <= 200
                    and positive_nodes >= 0 and positive_nodes.is_integer()):
                raise ValueError('Values are outside the expected ranges.')
            return pd.DataFrame([[age, operation_year, positive_nodes]], columns=X.columns)
        except ValueError as e:
            print('Invalid input:', e)
            if input('Do you want to start over? (y/n): ').strip().lower() != 'y':
                return None

# Run this cell to try an example prediction
new_data = get_new_data_point()
if new_data is not None:
    new_data_scaled = scaler.transform(new_data)
    prediction = model.predict(new_data_scaled)[0]
    print('Predicted class:', prediction)
    print('1 = survived 5+ years; 2 = died within 5 years')
    print('Educational exercise only; not for medical decision-making.')

### Explanation for a General Audience

This dataset contains information about 306 breast cancer surgery patients. It includes three features: age, year of surgery, and number of positive axillary lymph nodes.

SVM was used to predict whether a patient survived at least five years after surgery.

The model achieved approximately 66.1% accuracy on the test dataset. However, it had difficulty identifying patients who died within five years.

This means the model is not very reliable for predicting individual survival outcomes. More data or model improvements may be needed.